In [16]:
import os
import json
import pandas as pd
from maomao.utils.constants import *

#### Characterization of negative sequences by category of evidence
- This notebook analyzes the provenance and biological meaning of negative labels in the peptide dataset by linking each sequence to the type of experimental or curated evidence supporting its negative annotation.

- As input, it loads the curated dataset restricted to sequences classified as negative after label-consistency analysis, together with an external evidence table that categorizes negative datasets according to their annotation criteria (e.g., experimentally validated non-hemolytic peptides, inferred negatives, or decoy datasets). Only source-level label columns are retained to ensure a clean mapping between sequences and their originating datasets.

- For each negative sequence, the notebook identifies the contributing data sources and expands sequence–source relationships into individual records. These are then merged with the evidence metadata to associate each sequence with one or more negative evidence categories. The resulting information is aggregated into a sequence-by-evidence-category pivot table, explicitly encoding missing associations.

- Finally, summary statistics describing the distribution of negative evidence categories are computed and appended to the dataset metadata. Both the per-sequence evidence matrix and the updated metadata file are exported, providing transparency into the composition of the negative class and enabling downstream analyses of label reliability, dataset bias, and negative sampling strategies.

In [17]:
toxic_effect = "hemolytic" # Change for different toxic effects (e.g., toxic, neurotoxic, hemolytic, etc.)
integration_folder = f"../../processed_data/integrating_and_cleaning_data"

- Read data

In [18]:
df_evidence = (
    pd.read_excel("../../raw_data/evidence_negative_dataset.xlsx")
    .assign(task=lambda x: x["task"].str.lower())
    .loc[lambda x: x["task"].str.contains(f"{toxic_effect}", case=False, na=False)] # Filter data sources by activity

)
df_evidence

,name source,task,obtaining negative dataset,negative dataset category
0,BIOPEP-UWM,"celiac toxic, cytotoxic, hemolytic, toxic, emb...",No information,no negative data
1,CICERON,"celiac toxic, cytotoxic, hemolytic, embryotoxi...",No information,no negative data
2,Plantpepdb,"celiac toxic, cytotoxic, hemolytic, toxic, tox...",No information,no negative data
3,Peptipedia2.0,"cytotoxic, hemolytic, neurotoxic, toxic",No information,no negative data
4,MultiPep,"celiac toxic, hemolytic, embryotoxic, toxic, i...",No information,no negative data
19,DRAMP,"cytotoxic, cytolytic, toxic, insecticidal, hem...",Unrelated activities,weak or unconfirmed negatives
20,iAMPCN,"cytotoxic, hemolytic, insecticidal, toxic, ant...",Sampling from uniprot,weak or unconfirmed negatives
22,Hemolytik2.0_new,hemolytic,Experimentally validated,strong negatives
23,AMPDB,"cytotoxic, hemolytic, platelet aggregation inh...",No information,no negative data
26,Abdelbaky et al.,hemolytic,Sampling from Swiss-Prot,weak or unconfirmed negatives


In [19]:
df_only_negative = (
    pd.read_csv(f"{integration_folder}/{toxic_effect}/negative.csv")
      .loc[:, lambda df: ~df.columns.str.contains("unlabel")]
      .iloc[:, :-7]
)

In [20]:
df_only_negative

,sequence,Abdelbaky et al.,Almotairi et al.,AMPDB,AMPDeep,Bhatnagar et al.,BIOPEP-UWM,CICERON,ConsAMPHemo,DRAMP,...,Zhao et al.,QSVM-peptide,Plisson et al.,MultiTox,Peptipedia2.0,iAMPCN,Multipep,Hemolytik20_new,counts_1,counts_0
0,RDRLKDLGSEKIERLRGFQLSGQSDRIRRKILSEFGL,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,999,999,0,1
1,DDITKMLVATTH,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,999,999,0,1
2,GSPIQCAETCFIGKCYTEELGCTCTAFLCMKN,999,999,999,999,999,999,999,999,999,...,999,999,999,999,999,0,999,999,0,1
3,IGKSALDSRLGVFFVADQKGKGKSSGSAILVKLSAKF,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,999,999,0,1
4,AGFKRMWRV,999,999,999,999,999,999,999,999,999,...,999,999,999,999,999,0,999,999,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19752,YYFNAKKVKRRRWIWKK,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,999,999,0,1
19753,SEEALKALHESIIKAGYKPGEDIAIACDCA,0,0,999,0,999,999,999,0,999,...,999,999,999,999,999,0,999,999,0,6
19754,FLKLLKALL,999,999,999,999,999,999,999,999,999,...,999,999,999,999,999,999,999,999,0,3
19755,KSRIVPAIPVSLL,999,999,999,999,999,999,999,999,999,...,999,999,999,999,999,0,999,999,0,1


- Create dataset pivote

In [21]:
# Function to create the 'name source' column with the column names that have values ​​other than 999
def create_name_source(row):
    return [col for col in row.index[1:] if row[col] != 999]

# Exclude the 'sequence' column

# Apply the function row by row
df_only_negative['name source'] = df_only_negative.apply(create_name_source, axis=1)

# Expand the 'name source' lists into individual rows
df_exploded = df_only_negative.explode('name source').reset_index(drop=True)

# Final result
df_result = df_exploded[['sequence', 'name source']]

In [22]:
df_exploded = df_exploded.merge(df_evidence[['name source', 'negative dataset category']], on='name source', how='left')
df_exploded

,sequence,Abdelbaky et al.,Almotairi et al.,AMPDB,AMPDeep,Bhatnagar et al.,BIOPEP-UWM,CICERON,ConsAMPHemo,DRAMP,...,Plisson et al.,MultiTox,Peptipedia2.0,iAMPCN,Multipep,Hemolytik20_new,counts_1,counts_0,name source,negative dataset category
0,RDRLKDLGSEKIERLRGFQLSGQSDRIRRKILSEFGL,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,0,1,AMPDeep,weak or unconfirmed negatives
1,RDRLKDLGSEKIERLRGFQLSGQSDRIRRKILSEFGL,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,0,1,counts_1,NaN
2,RDRLKDLGSEKIERLRGFQLSGQSDRIRRKILSEFGL,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,0,1,counts_0,NaN
3,DDITKMLVATTH,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,0,1,AMPDeep,weak or unconfirmed negatives
4,DDITKMLVATTH,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,0,1,counts_1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79138,KSRIVPAIPVSLL,999,999,999,999,999,999,999,999,999,...,999,999,999,0,999,999,0,1,counts_1,NaN
79139,KSRIVPAIPVSLL,999,999,999,999,999,999,999,999,999,...,999,999,999,0,999,999,0,1,counts_0,NaN
79140,RQIKIWFQNRRMKWKA,999,999,999,999,999,999,999,999,999,...,999,999,999,0,999,999,0,1,iAMPCN,weak or unconfirmed negatives
79141,RQIKIWFQNRRMKWKA,999,999,999,999,999,999,999,999,999,...,999,999,999,0,999,999,0,1,counts_1,NaN


In [23]:
df_exploded["negative dataset category"].value_counts()

negative dataset category
weak or unconfirmed negatives    37859
strong negatives                  1760
Name: count, dtype: int64

In [24]:
df_unique_sequences = df_exploded[['sequence', 'negative dataset category']].drop_duplicates()

# Pivotar el DataFrame para que las secuencias sean filas y los toxicity target las columnas
df_pivote_evidence = df_unique_sequences.pivot_table(index='sequence', columns='negative dataset category', aggfunc='size', fill_value=999)

In [25]:
df_pivote_evidence

negative dataset category,strong negatives,weak or unconfirmed negatives
sequence,,
AAAAAAAAAGETS,999,1
AAAAAAAAAK,1,1
AAAAAAAIKMLMDLVNERIMALNKKAKK,999,1
AAAAARRRIRKQAHAHSK,999,1
AAAAGSVWGAVNYTSDCNGECKRRGYKGGYCGSFANVNCWCET,999,1
...,...,...
wrkkafw,999,1
wrkkwfw,999,1
wrkwkwrwr,999,1


- Working with metada

In [26]:
with open(f"{integration_folder}/{toxic_effect}/metadata.json", "r") as f:
    metadata = json.load(f)

In [27]:
evidence_counts = (
    df_pivote_evidence
    .replace(999, 0)
    .sum()
    .astype(int)
    .to_dict()
)

In [28]:
metadata["evidence_negative_dataset_statistics"] = {"category": evidence_counts}

- Exporting data

In [29]:
with open(f"{integration_folder}/{toxic_effect}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [30]:
df_pivote_evidence.to_csv(f"{integration_folder}/{toxic_effect}/sequence_negative_evidece.csv")